# 4 - Métadonnées d'interface

La table `metadata` est le **contrat entre la base et l'interface** (cf.
`specification-bdd.md`, §2.2). Au-delà du libellé, du type SQL et du statut catégoriel,
elle porte cinq champs d'UI, tous `VARCHAR` nullable (`NULL` par défaut) :

| Champ | Rôle |
|---|---|
| `unit` | Suffixe d'axe / tooltip (`"€"`, `"%"`, `"MW"`) |
| `display_format` | Chaîne [d3-format](https://d3js.org/d3-format) (`",.2f"`, `".0%"`) |
| `family` | Famille thématique (regroupement des variables dans les menus) |
| `description` | Aide contextuelle |
| `default_aggregation` | `SUM`, `AVG`, `MIN`, `MAX`, `COUNT`, `MEDIAN`, `MODE` — validé à l'écriture |

Ces champs appartiennent au **producteur de métadonnées** : un update de données ne les
écrase jamais. Ils se renseignent à la construction via `column_metadata`, et se corrigent
sur une base existante via `update_column_metadata`.

### Table des matières

0. [Importation des modules](#s0)
1. [Données synthétiques](#s1)
2. [`column_metadata` à la construction du schéma](#s2)
3. [Écriture dans un catalogue DuckLake](#s3)
4. [Corriger les champs d'UI sur une base existante](#s4)
5. [Un update de données n'écrase pas les champs d'UI](#s5)
6. [Cas d'erreur validés à l'écriture](#s6)

## 0. Importation des modules <a id="s0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import sys
import warnings

import polars as pl

# Ajout du chemin vers le package
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.operations import DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder, SchemaBuilder

## 1. Données synthétiques <a id="s1"></a>

In [ ]:
# Jeu de résultats minimal : une clé (date, region) et deux variables de valeur
df = pl.DataFrame(
    {
        "date": ["2026-01-01", "2026-01-01", "2026-01-02", "2026-01-02"],
        "region": ["Ile-de-France", "Bretagne", "Ile-de-France", "Bretagne"],
        "consumption_mwh": [1200.0, 640.5, 1310.2, 655.1],
        "load_factor": [0.42, 0.31, 0.45, 0.33],
    }
)
df

## 2. `column_metadata` à la construction du schéma <a id="s2"></a>

`SchemaBuilder.create_metadata_table` accepte un dictionnaire `column_metadata` qui associe
chaque nom de colonne à un sous-dictionnaire `{label, unit, display_format, family,
description, default_aggregation}` (toutes les clés sont optionnelles).

- Le paramètre `column_labels` existant reste utilisable ; si un `label` est fourni **à la
  fois** par `column_labels` et par `column_metadata`, c'est `column_metadata` qui l'emporte.
- `default_aggregation` est normalisé en majuscules et validé à l'écriture.
- Les colonnes non citées gardent des champs d'UI `NULL`.

In [ ]:
column_metadata = {
    "consumption_mwh": {
        "label": "Consommation",
        "unit": "MWh",
        "display_format": ",.0f",
        "family": "Énergie",
        "description": "Consommation électrique agrégée sur la maille régionale.",
        "default_aggregation": "sum",  # normalisé en 'SUM'
    },
    "load_factor": {
        "unit": "%",
        "display_format": ".0%",
        "family": "Énergie",
        "default_aggregation": "avg",
    },
}

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    schema_builder = SchemaBuilder(df=df, categorical_threshold=10)

df_metadata = schema_builder.create_metadata_table(column_metadata=column_metadata)
df_metadata.select(
    [
        "name",
        "label",
        "unit",
        "display_format",
        "family",
        "description",
        "default_aggregation",
    ]
).to_pandas()

`date` et `region` (non citées dans `column_metadata`) portent `None` sur tous les champs
d'UI, tandis que `consumption_mwh` a bien reçu le libellé de `column_metadata` et une
`default_aggregation` passée à `SUM`.

## 3. Écriture dans un catalogue DuckLake <a id="s3"></a>

`DuckLakeTablesBuilder.build_schema` propage `column_metadata` jusqu'au DDL de la table
`metadata` : les cinq colonnes d'UI sont créées en `VARCHAR` et alimentées en une écriture.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    builder = DuckLakeTablesBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["date", "region"],
        dataset_label="Consommation régionale",
    )

builder.build_schema(column_metadata=column_metadata)

# Relecture de la table metadata telle qu'elle est stockée
builder.conn.execute(
    """
    SELECT name, label, unit, display_format, family, default_aggregation
    FROM metadata
    ORDER BY name
    """
).pl()

In [ ]:
# Le DDL : les champs d'UI sont bien des colonnes VARCHAR nullable
builder.conn.execute("DESCRIBE metadata").pl()

## 4. Corriger les champs d'UI sur une base existante <a id="s4"></a>

`update_column_metadata(column, **fields)` renseigne ou corrige `label` et les cinq champs
d'UI sans reconstruire la base (un simple `UPDATE` + invalidation du cache). Un champ non
autorisé, une `default_aggregation` invalide ou une colonne absente de `metadata` lèvent une
`ValueError` explicite.

In [ ]:
updater = DatabaseUpdater(
    connection=builder.conn,
    categorical_threshold=10,
)

# Correction : libellé + description ajoutés pour load_factor, agrégation revue
updater.update_column_metadata(
    "load_factor",
    label="Facteur de charge",
    description="Rapport entre puissance moyenne et puissance installée.",
    default_aggregation="MEDIAN",
)

updater.conn.execute(
    """
    SELECT name, label, unit, description, default_aggregation
    FROM metadata
    WHERE name = 'load_factor'
    """
).pl()

## 5. Un update de données n'écrase pas les champs d'UI <a id="s5"></a>

Les champs d'UI n'appartiennent qu'au producteur de métadonnées. Lors d'un
`update_database` (nouvelles lignes, nouveau lot), la mise à jour de `metadata` ne touche
que les champs dérivés des données (`sql_type`, `is_categorical`) : les champs d'UI restent
intacts.

In [ ]:
new_rows = pl.DataFrame(
    {
        "date": ["2026-01-03", "2026-01-03"],
        "region": ["Ile-de-France", "Bretagne"],
        "consumption_mwh": [1288.0, 651.7],
        "load_factor": [0.44, 0.32],
    }
)

updater.update_database(
    update_df=new_rows,
    keep="first",
    use_transaction=False,
    compact_after_update=False,  # inutile sur une connexion in-memory d'illustration
)

# Les champs d'UI sont inchangés après l'insertion
updater.conn.execute(
    """
    SELECT name, label, unit, display_format, family, description, default_aggregation
    FROM metadata
    WHERE name IN ('consumption_mwh', 'load_factor')
    ORDER BY name
    """
).pl()

## 6. Cas d'erreur validés à l'écriture <a id="s6"></a>

Chaque garde-fou lève une `ValueError` explicite.

In [ ]:
# 6.1 Colonne inconnue dans column_metadata
try:
    schema_builder.create_metadata_table(column_metadata={"unknown_col": {"unit": "€"}})
except ValueError as exc:
    print("colonne inconnue     ->", exc)

# 6.2 Clé inconnue dans un sous-dictionnaire
try:
    schema_builder.create_metadata_table(
        column_metadata={"load_factor": {"unit": "%", "color": "red"}}
    )
except ValueError as exc:
    print("clé inconnue         ->", exc)

# 6.3 default_aggregation invalide
try:
    schema_builder.create_metadata_table(
        column_metadata={"load_factor": {"default_aggregation": "TOTAL"}}
    )
except ValueError as exc:
    print("agrégation invalide  ->", exc)

# 6.4 update_column_metadata sur une colonne absente de metadata
try:
    updater.update_column_metadata("not_a_column", unit="€")
except ValueError as exc:
    print("colonne absente      ->", exc)

# 6.5 update_column_metadata sur un champ non autorisé
try:
    updater.update_column_metadata("load_factor", sql_type="DOUBLE")
except ValueError as exc:
    print("champ non autorisé   ->", exc)